# Online Linear Regression

Least squares from a stream with `LinReg` from [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), which keeps running sums of cross-products and solves for the coefficients when asked, with equal or exponential weights.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, Plots

Random.seed!(1)  # simulated: y = 1 + 2x + noise, so the coefficients should come out near [1, 2]
X = [ones(10_000) randn(10_000)]  # an intercept column and one predictor
y = X * [1.0, 2.0] .+ randn(10_000)  # Replace X and y with your own.

## Coefficients From the Stream

`LinReg` adds no intercept, so X carries a column of ones. With equal weights the coefficients equal ordinary least squares on the same data.

In [ ]:
o = fit!(LinReg(), zip(eachrow(X), y))
println("Online coefficients: ", coef(o))
println("Batch coefficients:  ", X \ y)

## A Relationship That Changes

Halfway through, the slope rises from 2 to 3. Equal weights average the two regimes, while exponential weights with span 500 follow the new slope.

In [ ]:
slope = vcat(fill(2.0, 5_000), fill(3.0, 5_000))
y2 = 1.0 .+ slope .* X[:, 2] .+ randn(10_000)

equal = LinReg()
weighted = LinReg(weight=ExponentialWeight(500))
se = Float64[]; sw = Float64[]
for (t, (x_t, y_t)) in enumerate(zip(eachrow(X), y2))
    fit!(equal, (x_t, y_t)); fit!(weighted, (x_t, y_t))
    if t % 100 == 0
        push!(se, coef(equal)[2]); push!(sw, coef(weighted)[2])
    end
end
plot(100:100:10_000, [se sw], label=["Incremental" "Exponentially Weighted"], xlabel="Observations", ylabel="Slope")

## One-Step-Ahead Predictions

Each response is predicted from the coefficients so far, before the model sees it, so the errors are out of sample. `predict` uses the coefficients of the last call of `coef`.

In [ ]:
o = LinReg()
errors = Float64[]
for (x_t, y_t) in zip(eachrow(X), y)
    if nobs(o) > length(x_t)
        coef(o)
        push!(errors, y_t - predict(o, x_t))
    end
    fit!(o, (x_t, y_t))
end
println("Root mean squared one-step-ahead error: ", sqrt(mean(abs2, errors)))

## A Ridge Penalty

`coef(o, lambda)` adds lambda to the diagonal of the cross-products before solving, which shrinks the coefficients and steadies them when predictors are nearly collinear.

In [ ]:
println("No penalty:     ", coef(o))
println("Penalty 0.1:    ", coef(o, 0.1))
println("Penalty 1000.0: ", coef(o, 1000.0))